In [1]:
# ============================================================
# GIÁO TRÌNH: THỊ GIÁC MÁY: TỪ XỬ LÝ ẢNH ĐẾN HỌC SÂU
# BÀI CODE MINH HỌA: KHẢO SÁT BỘ LỌC ĐẦU TIÊN VÀ BIỂU DIỄN ĐẶC TRƯNG CỦA VGG-19
# Chương/Mục liên quan: Chương 7 - Mạng nơ-ron tích chập và biểu diễn đặc trưng học sâu
# ============================================================

# ============================================================
# MÔ TẢ
# ============================================================

# Mục đích:
# - Minh họa cách các kernel ở lớp tích chập đầu tiên của VGG-19 có thể gần với
#   một số bộ lọc cổ điển như Gaussian, Sobel và Laplacian.
# - Minh họa feature maps ở lớp đầu, Grad-CAM ở lớp tích chập cuối và đặc trưng theo block.
# - Giúp người học liên hệ giữa bộ lọc thủ công trong xử lý ảnh và bộ lọc học được trong CNN.

# Sau khi chạy code, người học cần:
# 1. Quan sát được hình dạng kernel học được ở lớp conv đầu tiên.
# 2. Hiểu cách dùng cosine similarity để so sánh kernel CNN với bộ lọc cổ điển.
# 3. Quan sát được sự thay đổi feature maps từ block nông đến block sâu.
# 4. Hiểu Grad-CAM cho biết vùng ảnh đóng góp mạnh vào dự đoán của mạng.

# Input:
# - Dữ liệu đầu vào: ảnh RGB từ GitHub.
# - Mô hình: VGG-19 pretrained trên ImageNet.

# Output:
# - Ảnh đầu vào.
# - Các kernel CNN gần với Gaussian, Sobel-x, Sobel-y và Laplacian.
# - Feature maps tương ứng.
# - Grad-CAM cho lớp dự đoán mạnh nhất.
# - Top-k feature maps và bản đồ kích hoạt trung bình theo từng block.

# Lưu ý
# Đoạn code này được xây dựng với sự hỗ trợ của công cụ AI.
# Giảng viên đã đọc, kiểm tra và hiệu chỉnh nhằm bảo đảm tính chính xác,
# tính sư phạm và sự phù hợp với nội dung lý thuyết trong giáo trình.

# ============================================================
# 1. CÀI ĐẶT VÀ IMPORT THƯ VIỆN
# ============================================================

import io
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import convolve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# ============================================================
# 2. CẤU HÌNH CHUNG
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

image_url = "https://github.com/lthavnu/cv-book/raw/main/images/caltech-101/panda/image_0020.jpg"

# ============================================================
# 3. CÁC HÀM TIỆN ÍCH CHUNG
# ============================================================

def replace_relu_inplace(module):
    # Đổi ReLU(inplace=True) thành ReLU(inplace=False)
    # để tránh lỗi khi dùng backward hook cho Grad-CAM.
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU):
            setattr(module, name, nn.ReLU(inplace=False))
        else:
            replace_relu_inplace(child)


def read_image_from_url(url):
    with urllib.request.urlopen(url) as response:
        image_data = response.read()
    image = Image.open(io.BytesIO(image_data)).convert("RGB")
    return image


def normalize_2d_kernel(K, eps=1e-8):
    K = np.asarray(K, dtype=np.float32)
    K = K - K.mean()
    n = np.sqrt((K ** 2).sum()) + eps
    return K / n


def kernel_to_gray_mean(kernel_rgb_3x3x3):
    # Chuyển kernel 3x3x3 về kernel xám 3x3 bằng cách lấy trung bình theo kênh màu.
    return kernel_rgb_3x3x3.mean(axis=0)


def cosine_similarity_2d(A, B, eps=1e-8):
    # Dòng này tương ứng với công thức cosine similarity giữa hai kernel:
    # cos(A,B) = <A,B> / (||A|| ||B||)
    A = normalize_2d_kernel(A, eps)
    B = normalize_2d_kernel(B, eps)
    return float((A * B).sum())


def minmax01(X, eps=1e-8):
    X = X.astype(np.float32)
    mn, mx = X.min(), X.max()
    return (X - mn) / (mx - mn + eps)


def get_contrasting_text_color(value):
    return "white" if value < 0.5 else "black"


def show_single_image(ax, img, title="", cmap=None):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


def show_kernel_values(ax, kernel_2d, title="", cmap="gray", fmt="{:.2f}"):
    ax.imshow(kernel_2d, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks(range(kernel_2d.shape[1]))
    ax.set_yticks(range(kernel_2d.shape[0]))
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    norm_kernel = minmax01(kernel_2d)

    for i in range(kernel_2d.shape[0]):
        for j in range(kernel_2d.shape[1]):
            value = kernel_2d[i, j]
            text_color = get_contrasting_text_color(norm_kernel[i, j])
            ax.text(
                j, i, fmt.format(value),
                ha="center", va="center",
                color=text_color, fontsize=10.5
            )


def overlay_heatmap_on_image(img_rgb_float01, heatmap_float01, alpha=0.40):
    cmap = plt.get_cmap("jet")
    heatmap_rgb = cmap(heatmap_float01)[..., :3]
    overlay = (1 - alpha) * img_rgb_float01 + alpha * heatmap_rgb
    overlay = np.clip(overlay, 0, 1)
    return overlay

# ============================================================
# 4. TẢI MÔ HÌNH VGG-19 PRETRAINED
# ============================================================

weights = models.VGG19_Weights.IMAGENET1K_V1
model = models.vgg19(weights=weights).to(device)
replace_relu_inplace(model)
model.eval()

first_conv = model.features[0]
W = first_conv.weight.detach().cpu().clone()

print("Shape kernel block1_conv1:", tuple(W.shape))

# ============================================================
# 5. TẠO CÁC MẪU BỘ LỌC CỔ ĐIỂN 3x3
# ============================================================

def create_reference_kernels():
    refs = {}

    refs["Gaussian"] = 1 / 4 * np.array([
        [1, 2, 1],
        [2, 4, 2],
        [1, 2, 1]
    ], dtype=np.float32)

    refs["Sobel-x"] = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=np.float32)

    refs["Sobel-y"] = np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1]
    ], dtype=np.float32)

    refs["Laplacian"] = np.array([
        [ 0, -1,  0],
        [-1,  4, -1],
        [ 0, -1,  0]
    ], dtype=np.float32)

    return refs


def find_best_matching_filters(W):
    # Tìm filter CNN có cosine similarity lớn nhất với từng bộ lọc cổ điển.
    W_np = W.numpy()
    refs = create_reference_kernels()

    results = {}

    for ref_name, ref_kernel in refs.items():
        best_idx = -1
        best_score = -1e9
        best_sign = 1

        for i in range(W_np.shape[0]):
            kernel_i = W_np[i]
            gray_i = kernel_to_gray_mean(kernel_i)

            s_pos = cosine_similarity_2d(gray_i, ref_kernel)
            s_neg = cosine_similarity_2d(gray_i, -ref_kernel)

            if s_pos >= s_neg:
                score = s_pos
                sign = 1
            else:
                score = s_neg
                sign = -1

            if score > best_score:
                best_score = score
                best_idx = i
                best_sign = sign

        results[ref_name] = {
            "filter_index": best_idx,
            "score": best_score,
            "sign": best_sign
        }

    return results


matching_results = find_best_matching_filters(W)

print("\n=== Kết quả tìm filter gần với các bộ lọc cổ điển ===")
for name, info in matching_results.items():
    print(
        f"{name:12s} -> filter #{info['filter_index']:2d}, "
        f"cosine similarity = {info['score']:.4f}, sign = {info['sign']:+d}"
    )

# ============================================================
# 6. HIỂN THỊ CÁC KERNEL TÌM ĐƯỢC
# ============================================================

def visualize_selected_kernels(W, matching_results):
    names = ["Gaussian", "Sobel-x", "Sobel-y", "Laplacian"]
    refs = create_reference_kernels()

    fig, axes = plt.subplots(len(names), 3, figsize=(5.0, 1.4 * len(names)))

    if len(names) == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, name in enumerate(names):
        idx = matching_results[name]["filter_index"]
        score = matching_results[name]["score"]
        sign = matching_results[name]["sign"]

        kernel = W[idx].numpy()
        kernel_mean = sign * kernel.mean(axis=0)

        ref_kernel = refs[name]

        show_kernel_values(
            axes[r, 0],
            ref_kernel,
            title=f"{name}",
            cmap="gray",
            fmt="{:.1f}"
        )

        show_kernel_values(
            axes[r, 1],
            kernel_mean,
            title=f"Conv1 #{idx}",
            cmap="gray",
            fmt="{:.2f}"
        )

        axes[r, 2].axis("off")
        axes[r, 2].text(
            0.5, 0.5,
            f"cosine\nsimilarity\n{score:.4f}",
            ha="center", va="center",
            fontsize=8,
            bbox=dict(
                boxstyle="round",
                facecolor="lightgray",
                alpha=0.6
            )
        )

    plt.tight_layout()
    plt.show()


visualize_selected_kernels(W, matching_results)

# ============================================================
# 7. TẢI ẢNH ĐẦU VÀO VÀ TIỀN XỬ LÝ THEO ImageNet
# ============================================================

pil_image = read_image_from_url(image_url)
print("Image size:", pil_image.size)

preprocess = weights.transforms()
x = preprocess(pil_image).unsqueeze(0).to(device)

img_rgb = np.array(pil_image).astype(np.float32) / 255.0

# ============================================================
# 8. HIỂN THỊ ẢNH ĐẦU VÀO
# ============================================================

plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.title("Ảnh đầu vào")
plt.axis("off")
plt.show()

# ============================================================
# 9. TRÍCH FEATURE MAPS Ở block1_conv1
# ============================================================

with torch.no_grad():
    conv1_out = first_conv(x)

conv1_out = conv1_out.detach().cpu()[0]


def visualize_feature_maps(img_rgb, conv1_out, matching_results):
    names = ["Gaussian", "Sobel-x", "Sobel-y", "Laplacian"]
    fig, axes = plt.subplots(len(names), 3, figsize=(9, 2.4 * len(names)))

    if len(names) == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, name in enumerate(names):
        idx = matching_results[name]["filter_index"]
        fmap = conv1_out[idx].numpy()

        # Dòng này tương ứng với phép kích hoạt ReLU: y = max(0, x)
        fmap_relu = np.maximum(fmap, 0)

        fmap_show = minmax01(fmap_relu)

        show_single_image(axes[r, 0], img_rgb, title=f"Ảnh đầu vào\n{name}")
        show_single_image(axes[r, 1], fmap, title=f"Feature map thô\nfilter #{idx}", cmap="gray")
        show_single_image(axes[r, 2], fmap_show, title="Feature map (ReLU + normalize)", cmap="gray")

    plt.tight_layout()
    plt.show()


visualize_feature_maps(img_rgb, conv1_out, matching_results)

# ============================================================
# 10. DỰ ĐOÁN NHÃN ẢNH
# ============================================================

with torch.no_grad():
    logits = model(x)

    # Dòng này tương ứng với hàm softmax để đổi logits thành xác suất.
    probs = torch.softmax(logits, dim=1)

    top_prob, top_idx = probs.max(dim=1)

pred_class = int(top_idx.item())
pred_score = float(top_prob.item())
categories = weights.meta["categories"]
pred_label = categories[pred_class]

print(f"\nDự đoán top-1: {pred_label} (p = {pred_score:.4f})")

# ============================================================
# 11. GRAD-CAM CHO LỚP TÍCH CHẬP CUỐI CÙNG
# ============================================================

target_layer = model.features[34]

activations = {}
gradients = {}


def forward_hook(module, inp, out):
    activations["value"] = out


def backward_hook(module, grad_in, grad_out):
    gradients["value"] = grad_out[0]


handle_f = target_layer.register_forward_hook(forward_hook)
handle_b = target_layer.register_full_backward_hook(backward_hook)

model.zero_grad()
logits = model(x)
pred_class = logits.argmax(dim=1).item()
score = logits[0, pred_class]

score.backward()

A = activations["value"][0]
G = gradients["value"][0]

# Dòng này tương ứng với trọng số Grad-CAM:
# alpha_k = trung bình không gian của gradient theo từng kênh k.
alpha = G.mean(dim=(1, 2), keepdim=True)

# Dòng này tương ứng với bản đồ Grad-CAM:
# CAM = ReLU(sum_k alpha_k A_k)
cam = (alpha * A).sum(dim=0)
cam = F.relu(cam)

cam = cam.unsqueeze(0).unsqueeze(0)
cam = F.interpolate(
    cam,
    size=pil_image.size[::-1],
    mode="bilinear",
    align_corners=False
)

cam = cam[0, 0].detach().cpu().numpy()
cam = minmax01(cam)

overlay = overlay_heatmap_on_image(img_rgb, cam, alpha=0.40)

handle_f.remove()
handle_b.remove()

# ============================================================
# 12. HIỂN THỊ GRAD-CAM
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_single_image(axes[0], img_rgb, title="Ảnh đầu vào")
show_single_image(axes[1], cam, title=f"Grad-CAM\nclass = {pred_label}", cmap="jet")
show_single_image(axes[2], overlay, title="Overlay")
plt.tight_layout()
plt.show()

# ============================================================
# 13. TỔNG HỢP FILTER ĐÃ CHỌN VÀ FEATURE MAPS
# ============================================================

names = ["Gaussian", "Sobel-x", "Sobel-y", "Laplacian"]
fig, axes = plt.subplots(2, len(names), figsize=(2.6 * len(names), 5.0))

for c, name in enumerate(names):
    idx = matching_results[name]["filter_index"]
    sign = matching_results[name]["sign"]

    kernel = W[idx].numpy()
    kernel_mean = sign * kernel.mean(axis=0)

    fmap = conv1_out[idx].numpy()
    fmap = minmax01(np.maximum(fmap, 0))

    show_kernel_values(
        axes[0, c],
        kernel_mean,
        title=f"{name}\nConv1 #{idx}",
        cmap="gray",
        fmt="{:.2f}"
    )

    show_single_image(
        axes[1, c],
        fmap,
        title="feature map",
        cmap="gray"
    )

plt.tight_layout()
plt.show()

# ============================================================
# 14. SO SÁNH HIỆU ỨNG LỌC CỔ ĐIỂN VÀ KERNEL CNN
# ============================================================

def apply_classical_filter_to_rgb(img_rgb_float01, kernel_2d):
    filtered_channels = np.zeros_like(img_rgb_float01)

    for i in range(3):
        # Dòng này tương ứng với phép tích chập ảnh với kernel 2D.
        filtered_channels[:, :, i] = convolve(
            img_rgb_float01[:, :, i],
            kernel_2d,
            mode="nearest"
        )

    mean_filtered_image = filtered_channels.mean(axis=2)
    mean_filtered_image = np.maximum(mean_filtered_image, 0)
    return minmax01(mean_filtered_image)


def apply_cnn_kernel_directly_to_rgb(img_rgb_float01, cnn_kernel_3x3x3):
    output_sum = np.zeros_like(img_rgb_float01[:, :, 0], dtype=np.float32)

    for i in range(3):
        kernel_2d_for_channel = cnn_kernel_3x3x3[i, :, :]

        # Dòng này áp dụng trực tiếp từng lát kernel CNN lên từng kênh màu.
        convolved_channel = convolve(
            img_rgb_float01[:, :, i],
            kernel_2d_for_channel,
            mode="nearest"
        )

        output_sum += convolved_channel

    output_sum = np.maximum(output_sum, 0)
    return minmax01(output_sum)


def visualize_filter_effects(img_rgb, conv1_out, matching_results, W):
    names = ["Gaussian", "Sobel-x", "Sobel-y", "Laplacian"]
    refs = create_reference_kernels()

    fig, axes = plt.subplots(len(names), 4, figsize=(13, 2.8 * len(names)))

    if len(names) == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, name in enumerate(names):
        ref_kernel_2d = refs[name]
        idx = matching_results[name]["filter_index"]

        cnn_kernel_3x3x3 = W[idx].numpy()

        show_single_image(axes[r, 0], img_rgb, title="Ảnh đầu vào")

        classical_output = apply_classical_filter_to_rgb(img_rgb, ref_kernel_2d)
        show_single_image(
            axes[r, 1],
            classical_output,
            title=f"Hiệu ứng {name} (cổ điển)",
            cmap="gray"
        )

        direct_cnn_output = apply_cnn_kernel_directly_to_rgb(img_rgb, cnn_kernel_3x3x3)
        show_single_image(
            axes[r, 2],
            direct_cnn_output,
            title=f"Hiệu ứng Conv1 #{idx}\n(áp dụng trực tiếp)",
            cmap="gray"
        )

        cnn_fmap = conv1_out[idx].numpy()
        cnn_fmap_show = minmax01(np.maximum(cnn_fmap, 0))
        show_single_image(
            axes[r, 3],
            cnn_fmap_show,
            title=f"Feature map Conv1 #{idx}\n(đầu ra mạng)",
            cmap="gray"
        )

    plt.tight_layout()
    plt.show()


visualize_filter_effects(img_rgb, conv1_out, matching_results, W)

# ============================================================
# 15. KHẢO SÁT FEATURE MAPS THEO TỪNG BLOCK CỦA VGG-19
# ============================================================

block_last_relu_indices = {
    "Block1": 3,
    "Block2": 8,
    "Block3": 17,
    "Block4": 26,
    "Block5": 35
}

block_activations = {}
hook_handles = []


def make_block_hook(block_name):
    def hook(module, inp, out):
        block_activations[block_name] = out.detach()
    return hook


for block_name, layer_idx in block_last_relu_indices.items():
    handle = model.features[layer_idx].register_forward_hook(make_block_hook(block_name))
    hook_handles.append(handle)

with torch.no_grad():
    _ = model(x)

for handle in hook_handles:
    handle.remove()

# ============================================================
# 16. CHỌN TOP-K FEATURE MAPS MẠNH NHẤT
# ============================================================

def select_topk_feature_maps(feature_tensor, top_k=4):
    feat = feature_tensor[0].detach().cpu().numpy()
    feat_relu = np.maximum(feat, 0.0)

    # Dòng này tính điểm kích hoạt của mỗi kênh bằng tổng giá trị sau ReLU.
    channel_scores = feat_relu.sum(axis=(1, 2))

    selected_indices = np.argsort(channel_scores)[::-1][:top_k]

    selected_maps = []
    selected_scores = []

    for idx in selected_indices:
        fmap = feat_relu[idx]
        fmap_show = minmax01(fmap)
        selected_maps.append(fmap_show)
        selected_scores.append(channel_scores[idx])

    return selected_indices, selected_maps, selected_scores

# ============================================================
# 17. HIỂN THỊ TOP-K FEATURE MAPS CỦA TỪNG BLOCK
# ============================================================

def visualize_topk_feature_maps_by_block(img_rgb, block_activations, top_k=4):
    block_names = ["Block1", "Block2", "Block3", "Block4", "Block5"]

    nrows = len(block_names)
    ncols = 1 + top_k

    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 2.8 * nrows))

    if nrows == 1:
        axes = np.expand_dims(axes, axis=0)

    for r, block_name in enumerate(block_names):
        feature_tensor = block_activations[block_name]
        selected_indices, selected_maps, selected_scores = select_topk_feature_maps(
            feature_tensor,
            top_k=top_k
        )

        show_single_image(
            axes[r, 0],
            img_rgb,
            title=f"{block_name}\nẢnh đầu vào"
        )

        for c_idx in range(top_k):
            idx = selected_indices[c_idx]
            score = selected_scores[c_idx]
            fmap_show = selected_maps[c_idx]

            show_single_image(
                axes[r, c_idx + 1],
                fmap_show,
                title=f"kênh #{idx}\nscore={score:.1f}",
                cmap="gray"
            )

    plt.tight_layout()
    plt.show()


visualize_topk_feature_maps_by_block(img_rgb, block_activations, top_k=4)

# ============================================================
# 18. HIỂN THỊ MỨC KÍCH HOẠT TRUNG BÌNH CỦA MỖI BLOCK
# ============================================================

def visualize_mean_activation_by_block(img_rgb, block_activations):
    block_names = ["Block1", "Block2", "Block3", "Block4", "Block5"]

    fig, axes = plt.subplots(
        1,
        len(block_names) + 1,
        figsize=(3.2 * (len(block_names) + 1), 4)
    )

    show_single_image(
        axes[0],
        img_rgb,
        title="Ảnh đầu vào"
    )

    for i, block_name in enumerate(block_names, start=1):
        feat = block_activations[block_name][0].detach().cpu().numpy()
        feat_relu = np.maximum(feat, 0.0)

        # Dòng này lấy trung bình theo kênh để tạo bản đồ kích hoạt đại diện cho block.
        mean_map = feat_relu.mean(axis=0)

        mean_map = minmax01(mean_map)

        show_single_image(
            axes[i],
            mean_map,
            title=f"{block_name}\ntrung bình các kênh",
            cmap="gray"
        )

    plt.tight_layout()
    plt.show()


visualize_mean_activation_by_block(img_rgb, block_activations)

# ============================================================
# 19. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n=== KIỂM TRA KẾT QUẢ ===")
print("Tensor ảnh đầu vào:", tuple(x.shape))
print("Số kernel ở block1_conv1:", W.shape[0])
print("Kích thước feature map conv1:", tuple(conv1_out.shape))
print("Nhãn dự đoán top-1:", pred_label)
print("Xác suất dự đoán top-1:", round(pred_score, 4))
print("Kích thước Grad-CAM:", cam.shape)

assert x.ndim == 4
assert W.shape[0] == 64
assert conv1_out.ndim == 3
assert cam.shape[0] == img_rgb.shape[0]
assert cam.shape[1] == img_rgb.shape[1]
assert len(block_activations) == 5

print("Kết quả kiểm tra: code đã chạy đúng các bước chính.")

# ============================================================
# 20. GỢI Ý THỬ NGHIỆM CHO NGƯỜI HỌC
# ============================================================

# 1. Thay đổi image_url để kiểm tra ảnh khác và quan sát Grad-CAM thay đổi thế nào.
# 2. Thay đổi top_k trong visualize_topk_feature_maps_by_block để xem nhiều hoặc ít feature maps hơn.
# 3. Thay đổi alpha trong overlay_heatmap_on_image để điều chỉnh độ đậm của bản đồ nhiệt Grad-CAM.

Output hidden; open in https://colab.research.google.com to view.